# 稠密連接網路（DenseNet）

ResNet極大地改變了如何參數化深層網路中函數的觀點。
*稠密連接網路*（DenseNet） :cite:`Huang.Liu.Van-Der-Maaten.ea.2017`在某種程度上是ResNet的邏輯擴展。讓我們先從數學上了解一下。

## 從ResNet到DenseNet

回想一下任意函數的泰勒展開式（Taylor expansion），它把這個函數分解成越來越高階的項。在$x$接近0時，

$$f(x) = f(0) + f'(0) x + \frac{f''(0)}{2!}  x^2 + \frac{f'''(0)}{3!}  x^3 + \ldots.$$

同樣，ResNet將函數展開為

$$f(\mathbf{x}) = \mathbf{x} + g(\mathbf{x}).$$

也就是說，ResNet將$f$分解為兩部分：一個簡單的線性項和一個複雜的非線性項。
那麼再向前拓展一步，如果我們想將$f$拓展成超過兩部分的信息呢？
一種方案便是DenseNet。

![ResNet（左）與 DenseNet（右）在跨層連接上的主要區別：使用相加和使用連結。](../img/densenet-block.svg)
:label:`fig_densenet_block`

如 :numref:`fig_densenet_block`所示，ResNet和DenseNet的關鍵區別在於，DenseNet輸出是*連結*（用圖中的$[,]$表示）而不是如ResNet的簡單相加。
因此，在應用越來越複雜的函數序列後，我們執行從$\mathbf{x}$到其展開式的映射：

$$\mathbf{x} \to \left[
\mathbf{x},
f_1(\mathbf{x}),
f_2([\mathbf{x}, f_1(\mathbf{x})]), f_3([\mathbf{x}, f_1(\mathbf{x}), f_2([\mathbf{x}, f_1(\mathbf{x})])]), \ldots\right].$$

最後，將這些展開式結合到多層感知機中，再次減少特徵的數量。
實現起來非常簡單：我們不需要添加術語，而是將它們連結起來。
DenseNet這個名字由變量之間的“稠密連接”而得來，最後一層與之前的所有層緊密相連。
稠密連接如 :numref:`fig_densenet`所示。

![稠密連接。](../img/densenet.svg)
:label:`fig_densenet`

稠密網路主要由2部分構成：*稠密塊*（dense block）和*過渡層*（transition layer）。
前者定義如何連接輸入和輸出，而後者則控制通道數量，使其不會太複雜。

## (**稠密塊體**)

DenseNet使用了ResNet改良版的“批量規範化、激活和卷積”架構（參見 :numref:`sec_resnet`中的練習）。
我們首先實現一下這個架構。


In [1]:
import torch
from torch import nn


def conv_block(input_channels, num_channels):
    return nn.Sequential(
        nn.BatchNorm2d(input_channels), nn.ReLU(),
        nn.Conv2d(input_channels, num_channels, kernel_size=3, padding=1))

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


一個*稠密塊*由多個卷積塊組成，每個卷積塊使用相同數量的輸出通道。
然而，在前向傳播中，我們將每個卷積塊的輸入和輸出在通道維上連結。


In [2]:
class DenseBlock(nn.Module):
    def __init__(self, num_convs, input_channels, num_channels):
        super(DenseBlock, self).__init__()
        layer = []
        for i in range(num_convs):
            layer.append(conv_block(
                num_channels * i + input_channels, num_channels))
        self.net = nn.Sequential(*layer)

    def forward(self, X):
        for blk in self.net:
            Y = blk(X)
            # 連接通道維度上每個塊的輸入和輸出
            X = torch.cat((X, Y), dim=1)
        return X

在下面的例子中，我們[**定義一個**]有2個輸出通道數為10的(**`DenseBlock`**)。
使用通道數為3的輸入時，我們會得到通道數為$3+2\times 10=23$的輸出。
卷積塊的通道數控制了輸出通道數相對於輸入通道數的增長，因此也稱為*增長率*（growth rate）。


In [3]:
blk = DenseBlock(2, 3, 10)
X = torch.randn(4, 3, 8, 8)
Y = blk(X)
Y.shape

torch.Size([4, 23, 8, 8])

## [**過渡層**]

由於每個稠密塊都會帶來通道數的增加，使用過多則會過於複雜化模型。
而過渡層可以用來控制模型複雜度。
它通過$1\times 1$卷積層來減小通道數，並使用步幅為2的平均彙集層減半高和寬，從而進一步降低模型複雜度。


In [4]:
def transition_block(input_channels, num_channels):
    return nn.Sequential(
        nn.BatchNorm2d(input_channels), nn.ReLU(),
        nn.Conv2d(input_channels, num_channels, kernel_size=1),
        nn.AvgPool2d(kernel_size=2, stride=2))

對上一個例子中稠密塊的輸出[**使用**]通道數為10的[**過渡層**]。
此時輸出的通道數減為10，高和寬均減半。


In [5]:
blk = transition_block(23, 10)
blk(Y).shape

torch.Size([4, 10, 4, 4])

## [**DenseNet模型**]

我們來構造DenseNet模型。DenseNet首先使用同ResNet一樣的單卷積層和最大彙集層。


In [6]:
b1 = nn.Sequential(
    nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3),
    nn.BatchNorm2d(64), nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2, padding=1))

接下來，類似於ResNet使用的4個殘差塊，DenseNet使用的是4個稠密塊。
與ResNet類似，我們可以設置每個稠密塊使用多少個卷積層。
這裡我們設成4，從而與 :numref:`sec_resnet`的ResNet-18保持一致。
稠密塊裡的卷積層通道數（即增長率）設為32，所以每個稠密塊將增加128個通道。

在每個模塊之間，ResNet通過步幅為2的殘差塊減小高和寬，DenseNet則使用過渡層來減半高和寬，並減半通道數。


In [7]:
# num_channels為當前的通道數
num_channels, growth_rate = 64, 32
num_convs_in_dense_blocks = [4, 4, 4, 4]
blks = []
for i, num_convs in enumerate(num_convs_in_dense_blocks):
    blks.append(DenseBlock(num_convs, num_channels, growth_rate))
    # 上一個稠密塊的輸出通道數
    num_channels += num_convs * growth_rate
    # 在稠密塊之間添加一個轉換層，使通道數量減半
    if i != len(num_convs_in_dense_blocks) - 1:
        blks.append(transition_block(num_channels, num_channels // 2))
        num_channels = num_channels // 2

與ResNet類似，最後接上全局彙集層和全連接層來輸出結果。


In [8]:
net = nn.Sequential(
    b1, *blks,
    nn.BatchNorm2d(num_channels), nn.ReLU(),
    nn.AdaptiveAvgPool2d((1, 1)),
    nn.Flatten(),
    nn.Linear(num_channels, 10))

## [**訓練模型**]

由於這裡使用了比較深的網路，本節裡我們將輸入高和寬從224降到96來簡化計算。


In [11]:
import torchvision
lr, num_epochs, batch_size = 0.1, 10, 256
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

# Load Fashion-MNIST dataset
from torchvision import transforms
transform = transforms.Compose([
    transforms.Resize(96),
    transforms.ToTensor()
])
train_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False)

# Move model to device
net = net.to(device)
optimizer = torch.optim.SGD(net.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

# Training loop
for epoch in range(num_epochs):
    net.train()
    train_loss = 0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = net(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    print(f'Epoch {epoch+1}/{num_epochs}:')
    print(f'Training Loss: {train_loss/len(train_loader):.3f}')
    print(f'Training Accuracy: {100.*correct/total:.2f}%')
    
    # Validation
    net.eval()
    test_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = net(images)
            loss = criterion(outputs, labels)
            
            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
    print(f'Test Loss: {test_loss/len(test_loader):.3f}')
    print(f'Test Accuracy: {100.*correct/total:.2f}%\n')

Epoch 1/10:
Training Loss: 0.510
Training Accuracy: 81.75%
Test Loss: 0.532
Test Accuracy: 81.20%

Epoch 2/10:
Training Loss: 0.285
Training Accuracy: 89.59%
Test Loss: 0.692
Test Accuracy: 74.84%

Epoch 3/10:
Training Loss: 0.240
Training Accuracy: 91.12%
Test Loss: 0.429
Test Accuracy: 83.98%

Epoch 4/10:
Training Loss: 0.213
Training Accuracy: 92.26%
Test Loss: 0.292
Test Accuracy: 89.26%

Epoch 5/10:
Training Loss: 0.194
Training Accuracy: 92.84%
Test Loss: 0.284
Test Accuracy: 90.04%

Epoch 6/10:
Training Loss: 0.177
Training Accuracy: 93.51%
Test Loss: 0.327
Test Accuracy: 88.46%

Epoch 7/10:
Training Loss: 0.164
Training Accuracy: 93.94%
Test Loss: 0.341
Test Accuracy: 87.78%

Epoch 8/10:
Training Loss: 0.152
Training Accuracy: 94.42%
Test Loss: 0.416
Test Accuracy: 87.29%

Epoch 9/10:
Training Loss: 0.142
Training Accuracy: 94.74%
Test Loss: 0.286
Test Accuracy: 90.06%

Epoch 10/10:
Training Loss: 0.130
Training Accuracy: 95.26%
Test Loss: 0.308
Test Accuracy: 89.39%



## 小結

* 在跨層連接上，不同於ResNet中將輸入與輸出相加，稠密連接網路（DenseNet）在通道維上連結輸入與輸出。
* DenseNet的主要構建模塊是稠密塊和過渡層。
* 在構建DenseNet時，我們需要通過添加過渡層來控制網路的維數，從而再次減少通道的數量。

## 練習

1. 為什麼我們在過渡層使用平均彙集層而不是最大彙集層？
1. DenseNet的優點之一是其模型參數比ResNet小。為什麼呢？
1. DenseNet一個詬病問題是記憶體或顯存消耗過多。
    1. 真的是這樣嗎？可以把輸入形狀換成$224 \times 224$，來看看實際的顯存消耗。
    1. 有另一種方法來減少顯存消耗嗎？需要改變框架嗎？
1. 實現DenseNet論文 :cite:`Huang.Liu.Van-Der-Maaten.ea.2017`表1所示的不同DenseNet版本。
1. 應用DenseNet的思想設計一個基於多層感知機的模型。將其應用於 :numref:`sec_kaggle_house`中的房價預測任務。


[Discussions](https://discuss.d2l.ai/t/1880)
